In [2]:
from nafnetlib import DeblurProcessor, DenoiseProcessor
import torch

In [5]:
from PIL import Image

In [7]:
db_processor = DenoiseProcessor(
    model_id="sidd_width32",
    model_dir="/Users/tjsss/Desktop/bharatAtomic/bA/lib/python3.9/site-packages/nafnetlib",
    device="mps"
)

# Load image for processing
img_path = "/Users/tjsss/Desktop/bharatAtomic/semPhase1/notebooks/outputs/2_1.png"
image = Image.open(img_path)

# Process the image (e.g., deblur or denoise)
db_processor.process(image).show()

In [1]:
%pip install diffusers torch accelerate

  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 4.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 5.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 6.1 MB/s  0:00:00 eta 0:00:01
Using cached click-8.1.8-py3-none-any.whl (98 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [accelerate]4 [accelerate]-hub]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from diffusers import DDIMPipeline

model_id = "google/ddpm-cifar10-32"

# load model and scheduler
ddim = DDIMPipeline.from_pretrained(model_id)

images = []
images
# run pipeline in inference (sample random noise and denoise)
image = ddim(num_inference_steps=50).images[0]

# save image
image.save("ddim_generated_image.png")

In [3]:
print(torch.cuda.memory_allocated()/1024**3 , 'used')
print(torch.cuda.memory_reserved()/1024**3 , 'reserved')
print(torch.cuda.max_memory_allocated()/1024**3 , 'peak')

0.0 used
0.0 reserved
0.0 peak


In [ ]:
##───────────────────────────────────────────────────Testing Individual Images───────────────────────────────────────────────────
def test_func(model, ip_img, transform, channels = 3, device=device):
    
    """
    channels=3 : model expects RGB input  (e.g. AttentionUNet, NAFNet trained on RGB)
    channels=1 : model expects grayscale  (e.g. RDUNet trained on single-channel SEM)
 
    transform   : an Albumentations Compose that at minimum resizes to the target size
                  (e.g. transform_3(256) or transform_1(256))
    """
    assert channels in (1, 3), "channels must be 1 or 3"

    device = device.type
    model.eval()
# ── 1. Load ────────────────────────────────────────────────────────────────
    with Image.open(ip_img) as img:
        if channels == 3:
            img_array = np.array(img.convert('RGB'))          # (H, W, 3) uint8
        else:
            img_array = np.array(img.convert('L'))            # (H, W)    uint8

# ── 2. Resize FIRST ────────────────────────────────────────────────────────
    if transform is not None:
        aug = transform(image=img_array)
        img_resized = aug["image"]                            # (256,256,3) or (256,256)
    else:
        img_resized = img_array
 
    noise_obj = NoiseImage()
    out = noise_obj.new_augment_sem(img_resized)              # handles 2D and 3D
 
    x_np = out["x"].astype(np.float32) / 255.0               # noisy
    y_np = out["y"].astype(np.float32) / 255.0               # clean

# ── 3. Numpy → Tensor ─────────────────────────────────────────────────────
    if channels == 3:
        # (H, W, 3) → (1, 3, H, W)
        x = torch.from_numpy(x_np).permute(2, 0, 1).unsqueeze(0).float().to(device)
        y = torch.from_numpy(y_np).permute(2, 0, 1).unsqueeze(0).float().to(device)
    else:
        # (H, W) → (1, 1, H, W)
        x = torch.from_numpy(x_np).unsqueeze(0).unsqueeze(0).float().to(device)
        y = torch.from_numpy(y_np).unsqueeze(0).unsqueeze(0).float().to(device)

# ── 4. Inference ──────────────────────────────────────────────────────────
    psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
 
    with torch.inference_mode():
        start = time.time()
        with torch.autocast(device, dtype=torch.float16):
            pred = model(x)
        pred_time = time.time() - start                       # pure model latency
 
        pred  = pred.clamp(0, 1).float()
        y_f   = y.float()
 
        ssim_score = ssim(pred, y_f, data_range=1.0, size_average=True).item()
        psnr_score = psnr_metric(pred, y_f).item()
        _, _, dists_val = all_losses(pred, y_f, train=False, c=channels)
        loss = calc_loss(pred, y_f, theta=0.4)
        pred_cpu = pred.cpu()
 
    total_time = time.time() - start
 
    # ── 5. Metrics ────────────────────────────────────────────────────────────
    print(f"Loss : {loss.item():.8f} | SSIM: {ssim_score:.4f}")
    print(f"PSNR : {psnr_score:.4f} dB | DISTS: {dists_val:.4f}")
    print(f"Pred Time: {pred_time:.4f}s  |  Total Time: {total_time:.4f}s")
 
    # ── 6. Visualisation (channel 0 for both 1ch and 3ch) ─────────────────────
    x_vis    = x.cpu()[0, 0].clamp(0, 1).numpy()
    pred_vis = pred_cpu[0, 0].clamp(0, 1).numpy()
    y_vis    = y_f.cpu()[0, 0].clamp(0, 1).numpy()
 
    plt.figure(figsize=(5, 5))
    plt.imshow(x_vis, cmap='gray')
    plt.title('Input (Degraded)')
    plt.axis('off')
    plt.show()
 
    plt.figure(figsize=(5, 5))
    plt.imshow(pred_vis, cmap='gray')
    plt.title('Prediction')
    plt.figtext(0.5, 0.02,
                f"Loss: {loss.item():.4f}, SSIM: {ssim_score:.4f}, PSNR: {psnr_score:.2f} dB",
                ha='center', fontsize=10)
    plt.axis('off')
    plt.show()
 
    plt.figure(figsize=(5, 5))
    plt.imshow(y_vis, cmap='gray')
    plt.title('Label (Clean)')
    plt.axis('off')
    plt.show()




##───────────────────────────────────────────────────Testing in Batches───────────────────────────────────────────────────
def test_func_batches(model, test_loader,channels = 3 ,device='cpu', transform = None):
    """
    channels=3 : model expects / outputs 3-channel tensors
    channels=1 : model expects / outputs 1-channel tensors
    """
    assert channels in (1, 3), "channels must be 1 or 3"


    model.eval()
    model.to(device)
    psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
    test_loss_sum = 0.0
    psnr_score = 0.0
    ssim_val, dists_val = 0.0,0.0

    with torch.no_grad():

        for batch_id, (x, y) in enumerate(test_loader):        
            x, y = x.to(device).float(), y.to(device).float()


            with torch.autocast('mps' , dtype = torch.float16):
                pred = model(x).to(device)
                test_loss = calc_loss(pred, y, theta=0.4)

            pred = pred.clamp(0.0, 1.0).float()

            
            ssim_b, psnr_b, dists_b = all_losses(pred, y)
            
            test_loss_sum += test_loss.item()
            ssim_sum      += ssim_b
            psnr_sum      += psnr_b
            dists_sum     += dists_b

            psnr_score += psnr_metric(pred, y).item()

            if (batch_id + 1) % 10 == 0:
                n = batch_id + 1
                print(
                    f"Batch {n}/{len(test_loader)} | "
                    f"Avg Loss: {test_loss_sum/n:.8f} | "
                    f"Avg PSNR: {psnr_sum/n:.4f} dB | "
                    f"Avg SSIM: {ssim_sum/n:.4f} | "
                    f"Avg DISTS: {dists_sum/n:.4f}"
                )
 
    n = len(test_loader)
    print(f"\n── Final Test Results ──────────────────")
    print(f"Avg Loss  : {test_loss_sum/n:.8f}")
    print(f"Avg PSNR  : {psnr_sum/n:.4f} dB")
    print(f"Avg SSIM  : {ssim_sum/n:.4f}")
    print(f"Avg DISTS : {dists_sum/n:.4f}")

            
